一、数据清洗

In [5]:
import pyarrow.parquet as pq
output_file = 'D:/workspace/Data_analysis_of_clothing_products/meta.parquet'
# 读取整个文件的前5行，查看数据有哪些标签
table = pq.read_table(output_file)
print(table.slice(0, 5).to_pandas())  

    main_category                                              title  \
0  AMAZON FASHION                   New Balance Men's 501 V1 Sneaker   
1  AMAZON FASHION               St Michael Antique Pendant Necklaces   
2  AMAZON FASHION                         Alpine Swiss Men's Sneaker   
3  AMAZON FASHION  norocos Women's Light Weight Shock Proof Slipp...   
4  AMAZON FASHION                Sam Edelman Women's Harlette Sandal   

   average_rating  rating_number  \
0             4.4             26   
1             5.0              2   
2             3.8            210   
3             4.1              8   
4             3.7             15   

                                            features  \
0  [100% Textile, Imported, Rubber sole, Textile ...   
1                                      [St. Michael]   
2  [Suede sole, Comes in Full Sizes Only - Half S...   
3                                                 []   
4  [100% Leather, Imported, Manmade sole, Strappy...   

             

In [ ]:
import pandas as pd
import numpy as np

# 读取整个数据集
output_file = 'D:/workspace/Data_analysis_of_clothing_products/meta.parquet'
df = pd.read_parquet(output_file)

# 查看数据集形状
print("数据集形状:", df.shape)

# 查看列名和数据类型
print("\n数据信息:")
df.info()

# 统计缺失值
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("\n各列缺失值数量（非零）:")
print(missing)

# 计算缺失比例
missing_ratio = (missing / len(df)) * 100
print("\n各列缺失比例（%）:")
print(missing_ratio)

# 显示前5行数据
print("\n数据前5行:")
df.head()

数据集形状: (50000, 14)

数据信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    49123 non-null  object 
 1   title            50000 non-null  object 
 2   average_rating   50000 non-null  float64
 3   rating_number    50000 non-null  int64  
 4   features         50000 non-null  object 
 5   description      50000 non-null  object 
 6   price            20858 non-null  float64
 7   images           50000 non-null  object 
 8   videos           50000 non-null  object 
 9   store            49848 non-null  object 
 10  categories       50000 non-null  object 
 11  details          50000 non-null  object 
 12  parent_asin      50000 non-null  object 
 13  bought_together  0 non-null      object 
dtypes: float64(2), int64(1), object(11)
memory usage: 5.3+ MB

各列缺失值数量（非零）:
bought_together    50000
price              29142
ma

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,AMAZON FASHION,New Balance Men's 501 V1 Sneaker,4.4,26,"[100% Textile, Imported, Rubber sole, Textile ...",[A casual sneaker with classic New Balance com...,NaN,[{'hi_res': 'https://m.media-amazon.com/images...,[],New Balance,"[Clothing, Shoes & Jewelry, Men, Shoes, Athlet...","{'ASTM Fluid Rating': None, 'Active Ingredient...",B0BRQ8PP11,None
1,AMAZON FASHION,St Michael Antique Pendant Necklaces,5.0,2,[St. Michael],[St. Michael is known in the Bible as the Arch...,NaN,"[{'hi_res': None, 'large': 'https://m.media-am...",[],St Michael,"[Clothing, Shoes & Jewelry, Women, Jewelry, Ne...","{'ASTM Fluid Rating': None, 'Active Ingredient...",B0056JNQMO,None
2,AMAZON FASHION,Alpine Swiss Men's Sneaker,3.8,210,"[Suede sole, Comes in Full Sizes Only - Half S...",[alpine Swiss Haris-BLK-13 Alpine Swiss Harris...,NaN,[{'hi_res': 'https://m.media-amazon.com/images...,[],Alpine Swiss,"[Clothing, Shoes & Jewelry, Men, Shoes, Fashio...","{'ASTM Fluid Rating': None, 'Active Ingredient...",B01C96XGJE,None
3,AMAZON FASHION,norocos Women's Light Weight Shock Proof Slipp...,4.1,8,[],[],NaN,[{'hi_res': 'https://m.media-amazon.com/images...,[],norocos,"[Clothing, Shoes & Jewelry, Women, Shoes, Sand...","{'ASTM Fluid Rating': None, 'Active Ingredient...",B06XSPZTZM,None
4,AMAZON FASHION,Sam Edelman Women's Harlette Sandal,3.7,15,"[100% Leather, Imported, Manmade sole, Strappy...",[Sam Edelman sparks stylish interest with the ...,NaN,[{'hi_res': 'https://m.media-amazon.com/images...,[],Circus by Sam Edelman,"[Clothing, Shoes & Jewelry, Women, Shoes, Sand...","{'ASTM Fluid Rating': None, 'Active Ingredient...",B008JCHHFC,None


In [7]:
df_clean = df.copy()

# 删除bought_together列（完全缺失）
df_clean.drop(columns=['bought_together'], inplace=True)

# 删除 main_category为空的行
df_clean.dropna(subset=['main_category'], inplace=True)

# 删除 store为空的行
df_clean.dropna(subset=['store'], inplace=True)

# 用同类商品均价填充 price 缺失
# 先计算每个 main_category 的价格均值（排除缺失值）
price_means = df_clean.groupby('main_category')['price'].transform('mean')
# 用均值填充缺失的price
df_clean['price'].fillna(price_means, inplace=True)

# 检查处理后的缺失情况
print("处理后缺失值统计：")
print(df_clean.isnull().sum())
print("\n数据集形状：", df_clean.shape)

处理后缺失值统计：
main_category      0
title              0
average_rating     0
rating_number      0
features           0
description        0
price             25
images             0
videos             0
store              0
categories         0
details            0
parent_asin        0
dtype: int64

数据集形状： (48980, 13)


C:\Users\ZhuanZ1\AppData\Local\Temp\ipykernel_9380\4276767679.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['price'].fillna(price_means, inplace=True)


In [8]:
# 删除 price 仍然为空的行（25条）
df_clean.dropna(subset=['price'], inplace=True)

# 再次检查缺失情况
print("删除 price 缺失后缺失值统计：")
print(df_clean.isnull().sum())
print("\n数据集形状：", df_clean.shape)

删除 price 缺失后缺失值统计：
main_category     0
title             0
average_rating    0
rating_number     0
features          0
description       0
price             0
images            0
videos            0
store             0
categories        0
details           0
parent_asin       0
dtype: int64

数据集形状： (48955, 13)


In [9]:
# 查看parent_asin重复情况
print("去重前数据形状:", df_clean.shape)
duplicated_count = df_clean['parent_asin'].duplicated().sum()
print("parent_asin 重复数量:", duplicated_count)

# 根据parent_asin去重，保留第一个出现的记录
df_clean.drop_duplicates(subset='parent_asin', keep='first', inplace=True)

# 重置索引
df_clean.reset_index(drop=True, inplace=True)

print("去重后数据形状:", df_clean.shape)
print("parent_asin 唯一值数量:", df_clean['parent_asin'].nunique())

去重前数据形状: (48955, 13)
parent_asin 重复数量: 0
去重后数据形状: (48955, 13)
parent_asin 唯一值数量: 48955


In [10]:
# 将清洗后的数据保存为 CSV 文件
output_csv = 'D:/workspace/Data_analysis_of_clothing_products/data_clean.csv'
df_clean.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"数据已保存至: {output_csv}")
print("文件大小:", df_clean.shape)

数据已保存至: D:/workspace/Data_analysis_of_clothing_products/data_clean.csv
文件大小: (48955, 13)
